In [1]:
import os
import numpy as np
import nibabel as nib
from nilearn.maskers import NiftiMasker
from scipy.io import loadmat
import seaborn as sns
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed
from joblib import Memory



In [2]:
cd /mnt/e/2026languageRSA

/mnt/e/2026languageRSA


In [ ]:
import os, sys, subprocess
print("kernel exe :", sys.executable)
print("kernel host:", subprocess.run(["hostname"], capture_output=True, text=True).stdout.strip()
      if os.name != "nt" else "WINDOWS")
print("cwd        :", os.getcwd())
for p in ["/mnt", "/mnt/e", "/mnt/e/2026languageRSA",
          "/mnt/e/2026languageRSA/epi", "/mnt/e/2026languageRSA/epi/part2"]:
    print(f"{os.path.isdir(p)!s:>5}  {p}")

In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Schaefer-400 (2 mm, Yeo-7) version with automatic subject-directory discovery.

Subject IDs are discovered by scanning the disk (MRI{number}{va|vb}); they are not assumed
to start at 01, be consecutive, or use zero padding. The row-to-subject mapping is written
to subject_ids in HDF5, which downstream code must use for indexing.
"""

import os
import re
import gc
import time
import numpy as np
import nibabel as nib
import h5py
from scipy.io import loadmat

from nilearn.maskers import NiftiMasker
from nilearn.datasets import fetch_atlas_schaefer_2018
from nilearn.image import resample_to_img, index_img, load_img
from nilearn.masking import apply_mask

# ==========================================
# Configuration
# ==========================================
NUM_RUNS = 6
ONSET_DELAY = 5

N_PARCELS = 400
YEO_NETWORKS = 17
RES_MM = 2

DUR_SPEAK = 16
DUR_IMAGERY = 6
N_TRIALS = 48                # 4 conditions x 12 trials
OUT_DTYPE = np.float32

INPUT_IMAGE_DIR = "/mnt/e/2026languageRSA/epi/"
TIMEINFO_PATH = "/mnt/e/2026languageRSA/timeinfo.mat"
ATLAS_CACHE = "/mnt/e/2026languageRSA/nilearn_data"

RUN_FILENAME = "run{run}.nii.gz"
SUBJECT_DIR_PAT = re.compile(r"^MRI(\d+)(va|vb)$", re.IGNORECASE)

OUT_FILES = {
    "va":  ("speak_va_400.h5",   DUR_SPEAK),
    "vb":  ("speak_vb_400.h5",   DUR_SPEAK),
    "var": ("imagery_va_400.h5", DUR_IMAGERY),
    "vbr": ("imagery_vb_400.h5", DUR_IMAGERY),
}


# ==========================================
# Subject discovery
# ==========================================
def discover_subjects(root):
    """
    Scan root for directories matching MRI{number}{va|vb}.
    Return (dirs, complete_ids, incomplete_ids).
    Sort IDs as integers so lexical ordering does not place MRI10 before MRI2.
    """
    if not os.path.isdir(root):
        raise FileNotFoundError(f"INPUT_IMAGE_DIR not found: {root}")

    dirs, unmatched = {}, []
    for name in sorted(os.listdir(root)):
        full = os.path.join(root, name)
        if not os.path.isdir(full):
            continue
        m = SUBJECT_DIR_PAT.match(name)
        if m is None:
            unmatched.append(name)
            continue
        dirs.setdefault(int(m.group(1)), {})[m.group(2).lower()] = full

    complete = sorted(s for s, v in dirs.items() if "va" in v and "vb" in v)
    incomplete = sorted(s for s, v in dirs.items() if len(v) < 2)

    if unmatched:
        print(f"  [discover] {len(unmatched)} dirs ignored: {unmatched[:8]}")
    if incomplete:
        print(f"  [discover] single-session subjects EXCLUDED: {incomplete}")
    return dirs, complete, incomplete


SUBJECT_DIRS, SUBJECT_IDS, _INCOMPLETE = discover_subjects(INPUT_IMAGE_DIR)
NUM_SUBJECTS = len(SUBJECT_IDS)
SUB_POS = {sub: i for i, sub in enumerate(SUBJECT_IDS)}
if NUM_SUBJECTS == 0:
    raise RuntimeError(f"No MRI**va/vb pairs found under {INPUT_IMAGE_DIR}")


def epi_path(sub, sess, run):
    """Return the path for this run, or None if it does not exist."""
    d = SUBJECT_DIRS.get(sub, {}).get(sess)
    if d is None:
        return None
    p = os.path.join(d, RUN_FILENAME.format(run=run))
    return p if os.path.exists(p) else None


# ==========================================
# Helpers
# ==========================================
def find_reference_image():
    for sub in SUBJECT_IDS:
        for sess in ("va", "vb"):
            for run in range(1, NUM_RUNS + 1):
                p = epi_path(sub, sess, run)
                if p:
                    print(f"Reference grid from: {p}")
                    return index_img(p, 0)
    raise FileNotFoundError("No EPI found under " + INPUT_IMAGE_DIR)


def build_atlas(ref_img):
    atlas = fetch_atlas_schaefer_2018(
        n_rois=N_PARCELS, yeo_networks=YEO_NETWORKS,
        resolution_mm=RES_MM, data_dir=ATLAS_CACHE,
    )
    atlas_img = load_img(atlas.maps)

    same_shape = tuple(atlas_img.shape[:3]) == tuple(ref_img.shape[:3])
    same_affine = np.allclose(atlas_img.affine, ref_img.affine, atol=1e-4)
    if not (same_shape and same_affine):
        print(f"Atlas grid {atlas_img.shape[:3]} != EPI grid {ref_img.shape[:3]} "
              f"-> resample_to_img(nearest)")
        try:
            atlas_img = resample_to_img(atlas_img, ref_img,
                                        interpolation="nearest",
                                        force_resample=True, copy_header=True)
        except TypeError:
            atlas_img = resample_to_img(atlas_img, ref_img,
                                        interpolation="nearest")
    else:
        print("Atlas grid already matches EPI grid; no resampling.")

    labels = []
    for l in np.asarray(atlas.labels).ravel():
        labels.append(l.decode() if isinstance(l, bytes) else str(l))
    if len(labels) == N_PARCELS + 1 and "background" in labels[0].lower():
        labels = labels[1:]
    return atlas_img, labels


def load_session(masker, sub, sess):
    """Return the (T, V_global) array for six runs, or None if any run is missing."""
    out = []
    for run in range(1, NUM_RUNS + 1):
        p = epi_path(sub, sess, run)
        if p is None:
            print(f"  MISSING: MRI{sub}{sess} run{run}")
            return None
        out.append(masker.transform(p).astype(OUT_DTYPE, copy=False))
    return out


def process_extracted_data(extracted_runs, tm_data, time_duration, condition_indices):
    """Slice trials by onset, concatenate, and reorder by condition. Return (48, dur, V)."""
    if not extracted_runs:
        return None

    subject_extracted = []
    for run_num, run_onsets in enumerate(tm_data):
        current_run_data = extracted_runs[run_num]
        run_extracted = []
        for onset in run_onsets:
            start = int(onset + ONSET_DELAY)
            end = int(onset + time_duration + ONSET_DELAY)
            if start >= 0 and end <= current_run_data.shape[0]:
                run_extracted.append(current_run_data[start:end, :])
        if run_extracted:
            subject_extracted.append(np.asarray(run_extracted))

    if not subject_extracted:
        return None

    n_per_run = {len(x) for x in subject_extracted}
    if len(n_per_run) != 1:
        print(f"  WARNING unequal trials across runs: {[len(x) for x in subject_extracted]}")
        return None

    data_array = np.asarray(subject_extracted)          # (Runs, Trials, Time, V)
    if data_array.ndim != 4:
        return None

    reshaped = data_array.reshape(-1, time_duration, data_array.shape[-1])
    if reshaped.shape[0] != N_TRIALS:
        print(f"  WARNING got {reshaped.shape[0]} trials, expected {N_TRIALS}")
        return None

    order = np.concatenate([np.asarray(condition_indices[i]) for i in range(4)])
    return reshaped[order]


# ==========================================
# Main
# ==========================================
if __name__ == "__main__":
    t_start = time.time()

    print(f"{NUM_SUBJECTS} subjects: {SUBJECT_IDS}")
    missing = [(s, sess, r) for s in SUBJECT_IDS for sess in ("va", "vb")
               for r in range(1, NUM_RUNS + 1) if epi_path(s, sess, r) is None]
    print(f"Runs found: {NUM_SUBJECTS * 12 - len(missing)}/{NUM_SUBJECTS * 12}")
    for m in missing:
        print("  MISSING:", m)

    # ---- 1. timing info ----
    timeinfo = loadmat(TIMEINFO_PATH)
    tm_vas, tm_var, tm_vbs, tm_vbr = [
        np.array([timeinfo['timeinfo'][name].squeeze().tolist()[:, i - 1]
                  for i in range(1, 7)])
        for name in ['vas_matrix', 'var_matrix', 'vbs_matrix', 'vbr_matrix']
    ]
    condition_indices = {i: [x for x in range(42 + 2 * i) if x % 8 in (i * 2, i * 2 + 1)]
                         for i in range(4)}
    assert sum(len(v) for v in condition_indices.values()) == N_TRIALS
    TM = {"va": tm_vas, "vb": tm_vbs, "var": tm_var, "vbr": tm_vbr}

    # ---- 2. atlas ----
    ref_img = find_reference_image()
    atlas_img, labels = build_atlas(ref_img)
    atlas_data = np.asarray(atlas_img.dataobj).astype(np.int16)

    mask_bool = atlas_data > 0
    mask_img = nib.Nifti1Image(mask_bool.astype(np.uint8), atlas_img.affine)
    n_vox_global = int(mask_bool.sum())
    print(f"Global cortical mask: {n_vox_global} voxels")

    masker = NiftiMasker(mask_img=mask_img, standardize='zscore_sample')
    masker.fit()

    labels_1d = np.squeeze(apply_mask(atlas_img, mask_img)).astype(np.int16)
    assert labels_1d.shape[0] == n_vox_global
    parcel_cols = {p: np.flatnonzero(labels_1d == p) for p in range(1, N_PARCELS + 1)}
    empty_parcels = [p for p, c in parcel_cols.items() if c.size == 0]
    if empty_parcels:
        print(f"WARNING: {len(empty_parcels)} empty parcels: {empty_parcels}")

    # ---- 2b. Validate run lengths ----
    lens = set()
    for s in SUBJECT_IDS:
        for sess in ("va", "vb"):
            for r in range(1, NUM_RUNS + 1):
                p = epi_path(s, sess, r)
                if p:
                    lens.add(nib.load(p).shape[3])
    need = int(max(tm.max() for tm in (tm_vas, tm_vbs, tm_var, tm_vbr))
               + max(DUR_SPEAK, DUR_IMAGERY) + ONSET_DELAY)
    print(f"Run lengths: {sorted(lens)} | required >= {need}")
    assert min(lens) >= need, f"run too short: {min(lens)} < {need}"

    # ---- 3. Initialize HDF5 ----
    files = {}
    for key, (fname, dur) in OUT_FILES.items():
        hf = h5py.File(fname, 'w')
        for p in range(1, N_PARCELS + 1):
            nv = parcel_cols[p].size
            if nv == 0:
                continue
            hf.create_dataset(
                f"subarray_{p - 1}",
                shape=(NUM_SUBJECTS, N_TRIALS, dur, nv),
                dtype=OUT_DTYPE,
                chunks=(1, N_TRIALS, dur, nv),
                compression="gzip", compression_opts=1,
                fillvalue=np.nan,
            )
        hf.create_dataset("subject_ids", data=np.array(SUBJECT_IDS, dtype=np.int32))
        hf.create_dataset("parcel_labels",
                          data=np.array(labels, dtype=h5py.string_dtype()))
        hf.create_dataset("parcel_n_voxels",
                          data=np.array([parcel_cols[p].size
                                         for p in range(1, N_PARCELS + 1)], dtype=np.int32))
        hf.create_dataset("subject_valid", data=np.zeros(NUM_SUBJECTS, dtype=np.uint8))
        hf.attrs["ONSET_DELAY"] = ONSET_DELAY
        hf.attrs["standardize"] = "zscore_sample"
        files[key] = hf

    # ---- 4. Process each subject ----
    for si, sub in enumerate(SUBJECT_IDS):
        t0 = time.time()
        print(f"\n=== [{si + 1}/{NUM_SUBJECTS}] MRI{sub} ===")

        ts = {"va": load_session(masker, sub, "va"),
              "vb": load_session(masker, sub, "vb")}

        globals_arr = {
            "va":  process_extracted_data(ts["va"], TM["va"],  DUR_SPEAK,   condition_indices) if ts["va"] else None,
            "vb":  process_extracted_data(ts["vb"], TM["vb"],  DUR_SPEAK,   condition_indices) if ts["vb"] else None,
            "var": process_extracted_data(ts["va"], TM["var"], DUR_IMAGERY, condition_indices) if ts["va"] else None,
            "vbr": process_extracted_data(ts["vb"], TM["vbr"], DUR_IMAGERY, condition_indices) if ts["vb"] else None,
        }
        del ts
        gc.collect()

        for key, hf in files.items():
            arr = globals_arr[key]
            if arr is None:
                print(f"  {key}: skipped")
                continue
            arr = arr.astype(OUT_DTYPE, copy=False)
            for p in range(1, N_PARCELS + 1):
                cols = parcel_cols[p]
                if cols.size == 0:
                    continue
                hf[f"subarray_{p - 1}"][si] = arr[:, :, cols]
            hf["subject_valid"][si] = 1

        del globals_arr
        gc.collect()
        print(f"  done in {time.time() - t0:.1f}s")

    for hf in files.values():
        hf.close()

    print(f"\nAll done in {(time.time() - t_start) / 60:.1f} min")

23 subjects: [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 17, 18, 19, 20, 24, 25, 26, 27, 28, 29, 30, 31]
Runs found: 276/276
Reference grid from: /mnt/e/2026languageRSA/epi/MRI01va/run1.nii.gz


[get_dataset_dir] Dataset found in /mnt/e/2026languageRSA/nilearn_data/schaefer_2018

[fetch_single_file] Downloading data from 
https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/b
rain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_400Parcels_17Networks_order.txt ...

[fetch_single_file]  ...done. (0 seconds, 0 min)

[fetch_single_file] Downloading data from 
https://raw.githubusercontent.com/ThomasYeoLab/CBIG/v0.14.3-Update_Yeo2011_Schaefer2018_labelname/stable_projects/b
rain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Schaefer2018_400Parcels_17Networks_order_FSLMNI152_2mm
.nii.gz ...

[fetch_single_file]  ...done. (0 seconds, 0 min)

Atlas grid already matches EPI grid; no resampling.
Global cortical mask: 132032 voxels
Run lengths: [604] | required >= 595

=== [1/23] MRI1 ===
  done in 955.1s

=== [2/23] MRI2 ===
  done in 958.4s

=== [3/23] MRI3 ===
  done in 952.2s

=== [4/23] MRI4 ===
  done in 975.0s

=== [5/23] MRI5 ===
  done in 873.2s

=== [6/23] MRI6 ===
  done in 958.6s

=== [7/23] MRI8 ===
  done in 907.2s

=== [8/23] MRI9 ===
  done in 957.5s

=== [9/23] MRI10 ===
  done in 896.8s

=== [10/23] MRI11 ===
  done in 912.3s

=== [11/23] MRI12 ===
  done in 899.8s

=== [12/23] MRI17 ===
  done in 951.0s

=== [13/23] MRI18 ===
  done in 909.4s

=== [14/23] MRI19 ===
  done in 951.5s

=== [15/23] MRI20 ===
  done in 1006.9s

=== [16/23] MRI24 ===
  done in 948.2s

=== [17/23] MRI25 ===
  done in 905.8s

=== [18/23] MRI26 ===
  done in 908.7s

=== [19/23] MRI27 ===
  done in 938.3s

=== [20/23] MRI28 ===
  done in 907.0s

=== [21/23] MRI29 ===
  done in 947.3s

=== [22/23] MRI30 ===
  done in 947.8s

=== [23/23

In [4]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Schaefer-400 (2 mm, Yeo-17) with automatic subject-directory discovery.
Generate only four response groups (the five TRs after the imagery window).

Slice interval = [onset + SHIFT + ONSET_DELAY, onset + SHIFT + DUR + ONSET_DELAY)
SHIFT=6, DUR=5, ONSET_DELAY=5 -> actual interval [onset+11, onset+16)

  response_vas : va images + vbr onset
  response_vbr : vb images + vbr onset
  response_var : va images + var onset
  response_vbs : vb images + var onset
"""

import os
import re
import gc
import time
import numpy as np
import nibabel as nib
import h5py
from scipy.io import loadmat

from nilearn.maskers import NiftiMasker
from nilearn.datasets import fetch_atlas_schaefer_2018
from nilearn.image import resample_to_img, index_img, load_img
from nilearn.masking import apply_mask

# ==========================================
# Configuration
# ==========================================
NUM_RUNS = 6
ONSET_DELAY = 5              # Hemodynamic delay in TRs

N_PARCELS = 400
YEO_NETWORKS = 17
RES_MM = 2

N_TRIALS = 48
OUT_DTYPE = np.float32

INPUT_IMAGE_DIR = "/mnt/e/2026languageRSA/epi/"
TIMEINFO_PATH = "/mnt/e/2026languageRSA/timeinfo.mat"
ATLAS_CACHE = "/mnt/e/2026languageRSA/nilearn_data"
OUT_DIR = "/mnt/e/2026languageRSA/derivatives/"

RUN_FILENAME = "run{run}.nii.gz"
SUBJECT_DIR_PAT = re.compile(r"^MRI(\d+)(va|vb)$", re.IGNORECASE)

RESP_SHIFT = 6
RESP_DUR = 5

# key: (session, onset matrix, SHIFT, DUR, file name)
OUTPUTS = {
    "response_vas": ("va", "vbr", RESP_SHIFT, RESP_DUR, "response_vas_400.h5"),
    "response_vbr": ("vb", "vbr", RESP_SHIFT, RESP_DUR, "response_vbr_400.h5"),
    "response_var": ("va", "var", RESP_SHIFT, RESP_DUR, "response_var_400.h5"),
    "response_vbs": ("vb", "var", RESP_SHIFT, RESP_DUR, "response_vbs_400.h5"),
}


# ==========================================
# Subject discovery
# ==========================================
def discover_subjects(root):
    if not os.path.isdir(root):
        raise FileNotFoundError(f"INPUT_IMAGE_DIR not found: {root}")

    dirs, unmatched = {}, []
    for name in sorted(os.listdir(root)):
        full = os.path.join(root, name)
        if not os.path.isdir(full):
            continue
        m = SUBJECT_DIR_PAT.match(name)
        if m is None:
            unmatched.append(name)
            continue
        dirs.setdefault(int(m.group(1)), {})[m.group(2).lower()] = full

    complete = sorted(s for s, v in dirs.items() if "va" in v and "vb" in v)
    incomplete = sorted(s for s, v in dirs.items() if len(v) < 2)

    if unmatched:
        print(f"  [discover] {len(unmatched)} dirs ignored: {unmatched[:8]}")
    if incomplete:
        print(f"  [discover] single-session subjects EXCLUDED: {incomplete}")
    return dirs, complete, incomplete


SUBJECT_DIRS, SUBJECT_IDS, _INCOMPLETE = discover_subjects(INPUT_IMAGE_DIR)
NUM_SUBJECTS = len(SUBJECT_IDS)
SUB_POS = {sub: i for i, sub in enumerate(SUBJECT_IDS)}
if NUM_SUBJECTS == 0:
    raise RuntimeError(f"No MRI**va/vb pairs found under {INPUT_IMAGE_DIR}")


def epi_path(sub, sess, run):
    d = SUBJECT_DIRS.get(sub, {}).get(sess)
    if d is None:
        return None
    p = os.path.join(d, RUN_FILENAME.format(run=run))
    return p if os.path.exists(p) else None


# ==========================================
# Helpers
# ==========================================
def find_reference_image():
    for sub in SUBJECT_IDS:
        for sess in ("va", "vb"):
            for run in range(1, NUM_RUNS + 1):
                p = epi_path(sub, sess, run)
                if p:
                    print(f"Reference grid from: {p}")
                    return index_img(p, 0)
    raise FileNotFoundError("No EPI found under " + INPUT_IMAGE_DIR)


def build_atlas(ref_img):
    atlas = fetch_atlas_schaefer_2018(
        n_rois=N_PARCELS, yeo_networks=YEO_NETWORKS,
        resolution_mm=RES_MM, data_dir=ATLAS_CACHE,
    )
    atlas_img = load_img(atlas.maps)

    same_shape = tuple(atlas_img.shape[:3]) == tuple(ref_img.shape[:3])
    same_affine = np.allclose(atlas_img.affine, ref_img.affine, atol=1e-4)
    if not (same_shape and same_affine):
        print(f"Atlas grid {atlas_img.shape[:3]} != EPI grid {ref_img.shape[:3]} "
              f"-> resample_to_img(nearest)")
        try:
            atlas_img = resample_to_img(atlas_img, ref_img, interpolation="nearest",
                                        force_resample=True, copy_header=True)
        except TypeError:
            atlas_img = resample_to_img(atlas_img, ref_img, interpolation="nearest")
    else:
        print("Atlas grid already matches EPI grid; no resampling.")

    labels = []
    for l in np.asarray(atlas.labels).ravel():
        labels.append(l.decode() if isinstance(l, bytes) else str(l))
    if len(labels) == N_PARCELS + 1 and "background" in labels[0].lower():
        labels = labels[1:]
    return atlas_img, labels


def load_session(masker, sub, sess):
    out = []
    for run in range(1, NUM_RUNS + 1):
        p = epi_path(sub, sess, run)
        if p is None:
            print(f"  MISSING: MRI{sub}{sess} run{run}")
            return None
        out.append(masker.transform(p).astype(OUT_DTYPE, copy=False))
    return out


def process_extracted_data(extracted_runs, tm_data, shift, time_duration, order):
    if not extracted_runs:
        return None

    subject_extracted = []
    for run_num, run_onsets in enumerate(tm_data):
        current_run_data = extracted_runs[run_num]
        run_extracted = []
        for onset in run_onsets:
            start = int(onset + shift + ONSET_DELAY)
            end = start + time_duration
            if start >= 0 and end <= current_run_data.shape[0]:
                run_extracted.append(current_run_data[start:end, :])
        if run_extracted:
            subject_extracted.append(np.asarray(run_extracted))

    if not subject_extracted:
        return None
    if len({len(x) for x in subject_extracted}) != 1:
        print(f"  WARNING unequal trials across runs: {[len(x) for x in subject_extracted]}")
        return None

    data_array = np.asarray(subject_extracted)
    if data_array.ndim != 4:
        return None

    reshaped = data_array.reshape(-1, time_duration, data_array.shape[-1])
    if reshaped.shape[0] != N_TRIALS:
        print(f"  WARNING got {reshaped.shape[0]} trials, expected {N_TRIALS}")
        return None
    return reshaped[order]


# ==========================================
# Main
# ==========================================
if __name__ == "__main__":
    t_start = time.time()
    if OUT_DIR not in (".", ""):
        os.makedirs(OUT_DIR, exist_ok=True)

    print(f"{NUM_SUBJECTS} subjects: {SUBJECT_IDS}")
    missing = [(s, sess, r) for s in SUBJECT_IDS for sess in ("va", "vb")
               for r in range(1, NUM_RUNS + 1) if epi_path(s, sess, r) is None]
    print(f"Runs found: {NUM_SUBJECTS * 12 - len(missing)}/{NUM_SUBJECTS * 12}")
    for m in missing:
        print("  MISSING:", m)

    # ---- 1. timing ----
    timeinfo = loadmat(TIMEINFO_PATH)
    TM = {}
    for short, name in [("vas", "vas_matrix"), ("var", "var_matrix"),
                        ("vbs", "vbs_matrix"), ("vbr", "vbr_matrix")]:
        TM[short] = np.array([timeinfo['timeinfo'][name].squeeze().tolist()[:, i - 1]
                              for i in range(1, 7)])
        print(f"  {short}: shape={TM[short].shape} "
              f"range=[{TM[short].min():.1f}, {TM[short].max():.1f}]")

    condition_indices = {i: [x for x in range(N_TRIALS) if x % 8 in (2 * i, 2 * i + 1)]
                         for i in range(4)}
    ORDER = np.concatenate([np.asarray(condition_indices[i]) for i in range(4)])
    assert sorted(ORDER.tolist()) == list(range(N_TRIALS)), "condition_indices does not cover 0..47"

    RUN_LABEL = np.tile(np.repeat(np.arange(1, NUM_RUNS + 1), 2), 4).astype(np.int32)
    COND_LABEL = np.repeat(np.arange(4), 12).astype(np.int32)

    # ---- 2. atlas ----
    ref_img = find_reference_image()
    atlas_img, labels = build_atlas(ref_img)
    atlas_data = np.asarray(atlas_img.dataobj).astype(np.int16)

    mask_bool = atlas_data > 0
    mask_img = nib.Nifti1Image(mask_bool.astype(np.uint8), atlas_img.affine)
    n_vox_global = int(mask_bool.sum())
    print(f"Global cortical mask: {n_vox_global} voxels")

    masker = NiftiMasker(mask_img=mask_img, standardize='zscore_sample')
    masker.fit()

    labels_1d = np.squeeze(apply_mask(atlas_img, mask_img)).astype(np.int16)
    assert labels_1d.shape[0] == n_vox_global
    parcel_cols = {p: np.flatnonzero(labels_1d == p) for p in range(1, N_PARCELS + 1)}
    empty_parcels = [p for p, c in parcel_cols.items() if c.size == 0]
    if empty_parcels:
        print(f"WARNING: {len(empty_parcels)} empty parcels: {empty_parcels}")

    # ---- 2b. Run lengths ----
    lens = set()
    for s in SUBJECT_IDS:
        for sess in ("va", "vb"):
            for r in range(1, NUM_RUNS + 1):
                p = epi_path(s, sess, r)
                if p:
                    try:
                        lens.add(nib.load(p).shape[3])
                    except Exception:
                        print(f"  unreadable: {p}")
    need = 0
    for key, (sess, tmname, shift, dur, _) in OUTPUTS.items():
        n = int(TM[tmname].max() + shift + dur + ONSET_DELAY)
        print(f"  {key:14s} {sess} + {tmname} +{shift} x{dur}  -> needs T >= {n}")
        need = max(need, n)
    print(f"Run lengths: {sorted(lens)} | required >= {need}")
    assert lens and min(lens) >= need, f"run too short: {min(lens) if lens else None} < {need}"

    # ---- 3. Initialize HDF5 (four files) ----
    files = {}
    for key, (sess, tmname, shift, dur, fname) in OUTPUTS.items():
        path = os.path.join(OUT_DIR, fname)
        if os.path.exists(path):
            raise FileExistsError(f"{path} already exists; move it first to avoid overwriting")
        hf = h5py.File(path, 'w')
        for p in range(1, N_PARCELS + 1):
            nv = parcel_cols[p].size
            if nv == 0:
                continue
            hf.create_dataset(f"subarray_{p - 1}",
                              shape=(NUM_SUBJECTS, N_TRIALS, dur, nv),
                              dtype=OUT_DTYPE, chunks=(1, N_TRIALS, dur, nv),
                              compression="gzip", compression_opts=1,
                              fillvalue=np.nan)
        hf.create_dataset("subject_ids", data=np.array(SUBJECT_IDS, dtype=np.int32))
        hf.create_dataset("subject_valid", data=np.zeros(NUM_SUBJECTS, dtype=np.uint8))
        hf.create_dataset("parcel_labels", data=np.array(labels, dtype=h5py.string_dtype()))
        hf.create_dataset("parcel_n_voxels",
                          data=np.array([parcel_cols[p].size
                                         for p in range(1, N_PARCELS + 1)], dtype=np.int32))
        hf.create_dataset("run_label", data=RUN_LABEL)
        hf.create_dataset("cond_label", data=COND_LABEL)
        hf.attrs["image_session"] = sess
        hf.attrs["onset_matrix"] = tmname
        hf.attrs["onset_shift"] = shift
        hf.attrs["duration_tr"] = dur
        hf.attrs["ONSET_DELAY"] = ONSET_DELAY
        hf.attrs["slice_formula"] = (
            f"[onset+{shift}+{ONSET_DELAY}, onset+{shift}+{dur}+{ONSET_DELAY})")
        hf.attrs["standardize"] = "zscore_sample"
        files[key] = hf
    print(f"{len(files)} output files initialized")

    # ---- 4. Process each subject ----
    for si, sub in enumerate(SUBJECT_IDS):
        t0 = time.time()
        print(f"\n=== [{si + 1}/{NUM_SUBJECTS}] MRI{sub} (row {si}) ===")

        ts = {"va": load_session(masker, sub, "va"),
              "vb": load_session(masker, sub, "vb")}

        for key, (sess, tmname, shift, dur, _) in OUTPUTS.items():
            if ts[sess] is None:
                continue
            arr = process_extracted_data(ts[sess], TM[tmname], shift, dur, ORDER)
            if arr is None:
                print(f"  {key}: skipped")
                continue
            arr = arr.astype(OUT_DTYPE, copy=False)
            hf = files[key]
            for p in range(1, N_PARCELS + 1):
                cols = parcel_cols[p]
                if cols.size:
                    hf[f"subarray_{p - 1}"][si] = arr[:, :, cols]
            hf["subject_valid"][si] = 1
            hf.flush()
            del arr

        del ts
        gc.collect()
        print(f"  done in {time.time() - t0:.1f}s")

    for hf in files.values():
        hf.close()

    print(f"\nAll done in {(time.time() - t_start) / 60:.1f} min")

23 subjects: [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 17, 18, 19, 20, 24, 25, 26, 27, 28, 29, 30, 31]
Runs found: 276/276
  vas: shape=(6, 8) range=[13.0, 558.0]
  var: shape=(6, 8) range=[29.0, 574.0]
  vbs: shape=(6, 8) range=[13.0, 558.0]
  vbr: shape=(6, 8) range=[29.0, 574.0]
Reference grid from: /mnt/e/2026languageRSA/epi/MRI01va/run1.nii.gz


[get_dataset_dir] Dataset found in /mnt/e/2026languageRSA/nilearn_data/schaefer_2018

Atlas grid already matches EPI grid; no resampling.
Global cortical mask: 132032 voxels
  response_vas   va + vbr +6 x5  -> needs T >= 590
  response_vbr   vb + vbr +6 x5  -> needs T >= 590
  response_var   va + var +6 x5  -> needs T >= 590
  response_vbs   vb + var +6 x5  -> needs T >= 590
Run lengths: [604] | required >= 590
4 output files initialized

=== [1/23] MRI1 (row 0) ===
  done in 939.0s

=== [2/23] MRI2 (row 1) ===
  done in 868.2s

=== [3/23] MRI3 (row 2) ===
  done in 864.4s

=== [4/23] MRI4 (row 3) ===
  done in 828.3s

=== [5/23] MRI5 (row 4) ===
  done in 787.9s

=== [6/23] MRI6 (row 5) ===
  done in 830.3s

=== [7/23] MRI8 (row 6) ===
  done in 818.4s

=== [8/23] MRI9 (row 7) ===
  done in 956.0s

=== [9/23] MRI10 (row 8) ===
  done in 934.2s

=== [10/23] MRI11 (row 9) ===
  done in 1003.5s

=== [11/23] MRI12 (row 10) ===
  done in 897.1s

=== [12/23] MRI17 (row 11) ===
  done in 1028.8s

=== [13/23] MRI18 (row 12) ===
  done in 1006.8s

=== [14/23] MRI19 (row 13) ===